### Coleta de dados sobre Dengue a partir do servidor FTP do DataSUS

ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/FINAIS

ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM

O arquivo que contem dados sobre a Dengue tem o prefixo DENGBR + ano .dbc, exemplo DENGBR25.dbc

In [1]:
import sys, os
from ftplib import FTP

import datasus_dbc
from dbfread import DBF
import pandas as pd


In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# Configurações de conexão e caminhos
ftp_host        = "ftp.datasus.gov.br"
pasta_remota    = "/dissemin/publicos/SINAN/DADOS/FINAIS/"
arquivo_remoto  = "DENGBR25.dbc"
arquivo_remoto  = "ZIKABR25.dbc"
arquivo_local   = "DENGBR25.dbc"
arquivo_local   = "ZIKABR25.dbc"

print(os.getcwd())

try:
    # Conecta ao servidor FTP de forma anônima
    print(f"Conectando em {ftp_host}...")
    ftp = FTP(ftp_host)
    ftp.login()  # Login anônimo padrão
    
    # Navega até a pasta correta
    ftp.cwd(pasta_remota)
    
    # Baixa o arquivo em modo binário
    print(f"Baixando {arquivo_remoto}...")
    with open(arquivo_local, "wb") as f:
        ftp.retrbinary(f"RETR {arquivo_remoto}", f.write)
        
    print(f"Download concluído com sucesso! Salvo como: {arquivo_local}")

except Exception as e:
    print(f"Ocorreu um erro ao baixar o arquivo: {e}")

finally:
    # Garante o fechamento da conexão
    try:
        ftp.quit()
    except:
        pass


c:\Marco Conti\Projetos\MAIS-v2\Data_SUS
Conectando em ftp.datasus.gov.br...
Baixando ZIKABR25.dbc...
Download concluído com sucesso! Salvo como: ZIKABR25.dbc


In [ ]:
path_arquivo_local = f"C:\Marco Conti\Projetos\MAIS-v2\Data_SUS\ZIKABR25.dbc"

# 1. Descompacta o arquivo .dbc gerando um arquivo .dbf temporário
datasus_dbc.decompress(path_arquivo_local, "arquivo_temporario_ZIKA.dbf")

# 2. Lê o arquivo .dbf gerado e converte para DataFrame do Pandas
tabela = DBF("arquivo_temporario.dbf", encoding="iso-8859-1") # DATASUS geralmente usa iso-8859-1 ou cp1252
df = pd.DataFrame(iter(tabela))

# Visualiza os dados
print(df.head())
